[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module5/04-classification.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module5/04-classification.ipynb)

# Module 5 — Lesson 4: Classification

**Module:** 5 — Machine Learning Foundations | **Time:** 45 minutes

## Learning Objectives

By the end of this lesson you will be able to:

- Train logistic regression and interpret its decision boundary
- Build and visualise decision trees, controlling depth to prevent overfitting
- Use `RandomForestClassifier` and interpret feature importances
- Apply Support Vector Machines with linear and RBF kernels
- Compare multiple classifiers using a benchmark table
- Produce classification reports and confusion matrix heatmaps

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.datasets import load_iris, load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score)

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('Libraries loaded.')

## 1. Logistic Regression and Decision Boundary

Logistic Regression applies the logistic (sigmoid) function to a linear combination of features:

```
p(y=1|x) = σ(β₀ + β₁x₁ + ... + βₙxₙ)
```

The decision boundary is the hyperplane where p = 0.5. We visualise it on the 2-D Iris dataset (petal length vs petal width).

In [ ]:
iris = load_iris()
# Use only 2 features and 2 classes for 2-D boundary visualisation
X_2d = iris.data[iris.target != 0, 2:4]  # petal length/width
y_2d = (iris.target[iris.target != 0] == 2).astype(int)  # versicolor=0, virginica=1

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_2d, y_2d, test_size=0.25, random_state=42)

lr = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression())])
lr.fit(X_tr2, y_tr2)

# Plot decision boundary
def plot_decision_boundary(model, X, y, ax, title):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlBu', edgecolors='k', s=40)
    ax.set_xlabel('Petal Length')
    ax.set_ylabel('Petal Width')
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_decision_boundary(lr, X_tr2, y_tr2, axes[0], f'Logistic Regression — Train (Acc={accuracy_score(y_tr2, lr.predict(X_tr2)):.2f})')
plot_decision_boundary(lr, X_te2, y_te2, axes[1], f'Logistic Regression — Test (Acc={accuracy_score(y_te2, lr.predict(X_te2)):.2f})')
plt.tight_layout()
plt.show()

## 2. Decision Trees

Decision trees recursively split the feature space by selecting thresholds that maximise class purity (Gini impurity or entropy). `max_depth` limits the tree's complexity.

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

train_accs, test_accs = [], []
depths = range(1, 16)
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, dt.predict(X_train)))
    test_accs.append(accuracy_score(y_test,  dt.predict(X_test)))

best_depth = depths[np.argmax(test_accs)]
print(f'Best max_depth for test accuracy: {best_depth} -> Test acc: {max(test_accs):.4f}')

plt.figure(figsize=(9, 4))
plt.plot(depths, train_accs, 'o-', label='Train', color='steelblue')
plt.plot(depths, test_accs, 's-', label='Test', color='tomato')
plt.axvline(best_depth, linestyle='--', color='green', label=f'Best depth={best_depth}')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Decision Tree: Depth vs Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Visualise the best tree (truncated to depth 3 for readability)
dt_best = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
dt_best.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(18, 6))
plot_tree(dt_best, max_depth=3, feature_names=data.feature_names,
          class_names=data.target_names, filled=True, ax=ax, fontsize=8)
ax.set_title(f'Decision Tree (max_depth={best_depth}, displayed to depth 3)', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Random Forest and Feature Importances

Random Forest builds many decision trees on bootstrap samples of the data with a random subset of features at each split. The ensemble's prediction is the majority vote (classification) or average (regression). Feature importances measure how much each feature reduces impurity across all trees.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print(f'Random Forest Test Accuracy: {accuracy_score(y_test, rf.predict(X_test)):.4f}')

importances = rf.feature_importances_
indices     = np.argsort(importances)[::-1]
top_n = 10

plt.figure(figsize=(10, 4))
plt.bar(range(top_n), importances[indices[:top_n]], color='steelblue', alpha=0.8)
plt.xticks(range(top_n), [data.feature_names[i] for i in indices[:top_n]], rotation=35, ha='right')
plt.ylabel('Mean Impurity Decrease')
plt.title('Random Forest — Top 10 Feature Importances')
plt.tight_layout()
plt.show()

print('\nTop 5 features:')
for rank, idx in enumerate(indices[:5], 1):
    print(f'  {rank}. {data.feature_names[idx]:40s} {importances[idx]:.4f}')

## 4. Support Vector Machines

SVMs find the maximum-margin hyperplane separating classes. The **kernel trick** maps data into higher-dimensional spaces implicitly:

- `linear` — good for linearly separable, high-dimensional data (e.g., text)
- `rbf` (Radial Basis Function) — general-purpose, handles non-linear boundaries

In [ ]:
svm_results = {}
for kernel in ['linear', 'rbf', 'poly']:
    pipe_svm = Pipeline([
        ('sc', StandardScaler()),
        ('svm', SVC(kernel=kernel, C=1.0, probability=True, random_state=42))
    ])
    cv_scores = cross_val_score(pipe_svm, X, y, cv=5, scoring='accuracy')
    pipe_svm.fit(X_train, y_train)
    svm_results[kernel] = {
        'cv_mean': cv_scores.mean(),
        'cv_std':  cv_scores.std(),
        'test_acc': accuracy_score(y_test, pipe_svm.predict(X_test))
    }
    print(f'SVC kernel={kernel:6s}: CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}  Test={svm_results[kernel]["test_acc"]:.4f}')

# Visualise boundaries on 2-D iris data
pipe_rbf = Pipeline([('sc', StandardScaler()),
                     ('svm', SVC(kernel='rbf', C=1.0))])
pipe_lin = Pipeline([('sc', StandardScaler()),
                     ('svm', SVC(kernel='linear', C=1.0))])
pipe_rbf.fit(X_tr2, y_tr2)
pipe_lin.fit(X_tr2, y_tr2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_decision_boundary(pipe_lin, X_2d, y_2d, axes[0], f'SVC linear (Acc={accuracy_score(y_2d, pipe_lin.predict(X_2d)):.2f})')
plot_decision_boundary(pipe_rbf, X_2d, y_2d, axes[1], f'SVC RBF (Acc={accuracy_score(y_2d, pipe_rbf.predict(X_2d)):.2f})')
plt.tight_layout()
plt.show()

## 5. Gradient Boosting Classifier

Gradient Boosting builds trees sequentially, each correcting the errors of the previous ensemble. It is one of the strongest off-the-shelf algorithms for tabular data.

In [ ]:
gb = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)
gb_cv = cross_val_score(gb, X, y, cv=5, scoring='accuracy')
print(f'GradientBoosting CV: {gb_cv.mean():.4f} ± {gb_cv.std():.4f}')
print(f'GradientBoosting Test Accuracy: {accuracy_score(y_test, gb.predict(X_test)):.4f}')

## 6. Classifier Benchmark Table

In [ ]:
classifiers = {
    'Logistic Regression': Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))]),
    'Decision Tree (d=5)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'SVC (linear)':        Pipeline([('sc', StandardScaler()), ('clf', SVC(kernel='linear', C=1.0))]),
    'SVC (RBF)':           Pipeline([('sc', StandardScaler()), ('clf', SVC(kernel='rbf', C=1.0))]),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=42),
}

benchmark = []
for name, clf in classifiers.items():
    cv_sc = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
    clf.fit(X_train, y_train)
    test_acc = accuracy_score(y_test, clf.predict(X_test))
    benchmark.append({'Classifier': name,
                      'CV Mean': round(cv_sc.mean(), 4),
                      'CV Std':  round(cv_sc.std(), 4),
                      'Test Acc': round(test_acc, 4)})

bench_df = pd.DataFrame(benchmark).sort_values('Test Acc', ascending=False).reset_index(drop=True)
print('=== Classifier Benchmark ===')
print(bench_df.to_string(index=False))

# Horizontal bar chart
plt.figure(figsize=(9, 4))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(bench_df)))
plt.barh(bench_df['Classifier'], bench_df['Test Acc'], color=colors[::-1], alpha=0.85)
plt.xlabel('Test Accuracy')
plt.title('Classifier Comparison — Breast Cancer Dataset')
plt.xlim(0.85, 1.0)
plt.tight_layout()
plt.show()

## 7. Classification Report and Confusion Matrix

In [ ]:
# Best model: Random Forest
best_clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
best_clf.fit(X_train, y_train)
y_pred = best_clf.predict(X_test)

print('=== Classification Report (Random Forest) ===')
print(classification_report(y_test, y_pred, target_names=data.target_names))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=data.target_names)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — Random Forest')

# Normalised confusion matrix
cm_norm = confusion_matrix(y_test, y_pred, normalize='true')
disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=data.target_names)
disp_norm.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Normalised Confusion Matrix')

plt.tight_layout()
plt.show()

## Practice Exercises

**Exercise 1 — Multi-class Classification**
Load the full `load_iris()` dataset (3 classes). Fit a `LogisticRegression(multi_class='ovr')` and a `LogisticRegression(multi_class='multinomial')`. Compare their test accuracy and classification reports. Which strategy performs better on Iris?

**Exercise 2 — Decision Tree Pruning**
Using `load_breast_cancer()`, find the optimal `max_depth` using 5-fold cross-validation instead of a single train/test split. Plot CV accuracy ± std for depths 1 to 20. Mark the best depth. Does it differ from the depth found in section 2?

**Exercise 3 — Custom Benchmark Extension**
Add `LogisticRegression` with `C=0.01` (heavily regularised) and `C=100` (barely regularised) as two separate entries to the benchmark table. Use cross-validation. What does the comparison reveal about the effect of regularisation on this dataset?